## Урок 3

### Задание начального уровня

Потренируйтесь в нахождении матрицы схожести для валидационного сета

* загрузите `brand_tweets_valid.csv`
* примените объект `vectorizer`, обученный на датасете `brand_tweets.csv` (просто скопируйте этот код из урока)
* примените функцию `pairwise_distances` к полученной матрице

In [3]:
import pandas as pd

df_valid = pd.read_csv('sample_data/brand_tweets_valid.csv', sep=',', encoding='utf8')
# удаляем строки, в которых отсутствует текст твита
df_valid.drop(df_valid[df_valid.tweet_text.isnull()].index, inplace=True)

print(df_valid.shape)

df_valid.head()
### YOUR CODE HERE ###



(402, 3)


,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,Wow! Google maps for mobile v5 demo at #sxsw. ...,Other Google product or service,Positive emotion
1,The #google name was built on gettinng stuff o...,Google,Positive emotion
2,&quot;Apple opening a temporary store in Austi...,NaN,No emotion toward brand or product
3,#tech Apple Opening Pop-Up Store In Austin For...,Apple,Positive emotion
4,GSDM Google party is off the hook! #SXSW {link},Google,Positive emotion


In [4]:
# стоп-слова
stop_words = [
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd",
    'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers',
    'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
    'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been',
    'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if',
    'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out',
    'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why',
    'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
    'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'shold',
    "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't",
    'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't",
    'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"
]

In [6]:
import nltk
import string
import pandas as pd
from itertools import chain
import numpy as np
nltk.download('punkt_tab')

# дополнительный словарь со знаками пунктуации
nltk.download('punkt', download_dir='.')
sample_str = df_valid.tweet_text.values[0]

print('== Исходный текст== \n%s\n\n' % sample_str)

tokenized_str = nltk.word_tokenize(sample_str)
print('== Токенизированный текст==\n%s' % tokenized_str)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to ....
[nltk_data]   Unzipping tokenizers/punkt.zip.


== Исходный текст== 
Wow! Google maps for mobile v5 demo at #sxsw. Very nice.


== Токенизированный текст==
['Wow', '!', 'Google', 'maps', 'for', 'mobile', 'v5', 'demo', 'at', '#', 'sxsw', '.', 'Very', 'nice', '.']


In [7]:
tokens = [i.lower() for i in tokenized_str if ( i not in string.punctuation )]
print(tokens)

['wow', 'google', 'maps', 'for', 'mobile', 'v5', 'demo', 'at', 'sxsw', 'very', 'nice']


In [8]:

def tokenize_text(raw_text: str):
    """Функция для токенизации текста

    :param raw_text: исходная текстовая строка
    """
    tokenized_str = nltk.word_tokenize(raw_text)
    tokens = [i.lower() for i in tokenized_str if ( i not in string.punctuation )]
    filtered_tokens = [i for i in tokens if ( i not in stop_words )]
    return filtered_tokens

# применяем функцию в датафрейму с помощью метода .apply()
tokenized_tweets= df_valid.tweet_text.apply(tokenize_text)

# добавляем новую колонку в исходный датафрейм
df = df_valid.assign(
    tokenized=tokenized_tweets
)

df.tokenized.head()

,tokenized
0,"[wow, google, maps, mobile, v5, demo, sxsw, nice]"
1,"[google, name, built, gettinng, stuff, trying,..."
2,"[quot, apple, opening, temporary, store, austi..."
3,"[tech, apple, opening, pop-up, store, austin, ..."
4,"[gsdm, google, party, hook, sxsw, link]"


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# инициализируем объект, который токенизирует наш текст
# в качестве единственного аргимента передаём функцию, которую мы написали в Уроке 2
# на разбивает каждый документ на токены
vectorizer = CountVectorizer(tokenizer=tokenize_text)
# применяем наш объект-токенизатор к датафрейму с твитами
document_matrix = vectorizer.fit_transform(df.tweet_text.values)
# результат - матрица, в которой находятся числа, строк в мастрице столько, сколько документов
# а столбцов столько, сколько токенов
document_matrix

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4927 stored elements and shape (402, 1760)>

In [10]:
source_tweet_index = 14
print(df.iloc[source_tweet_index].tweet_text)

Popup Apple Store crew has been giving out water to the people in line but they are in street clothes. No Apple logos anywhere yet. #SXSW


### Задание среднего уровня

Пользуясь матрицей схожести, полученной на предыдущем этапе, найдите top-5 твитов, похожих на твит валидационного сета с `id=14`.

In [11]:
test_tweet_index = 14

print(df_valid.iloc[test_tweet_index].tweet_text+'\n------------------------------\n')


### YOUR CODE HERE ###



Popup Apple Store crew has been giving out water to the people in line but they are in street clothes. No Apple logos anywhere yet. #SXSW
------------------------------



In [15]:
from sklearn.metrics import pairwise_distances

tweet_distance = 1-pairwise_distances(document_matrix, metric="cosine")

tweet_distance

array([[1.        , 0.21320072, 0.17149859, ..., 0.18898224, 0.16222142,
        0.21320072],
       [0.21320072, 1.        , 0.14625448, ..., 0.24174689, 0.20751434,
        0.18181818],
       [0.17149859, 0.14625448, 1.        , ..., 0.12964074, 0.27820744,
        0.14625448],
       ...,
       [0.18898224, 0.24174689, 0.12964074, ..., 1.        , 0.1839418 ,
        0.16116459],
       [0.16222142, 0.20751434, 0.27820744, ..., 0.1839418 , 1.        ,
        0.13834289],
       [0.21320072, 0.18181818, 0.14625448, ..., 0.16116459, 0.13834289,
        1.        ]])

In [18]:
import numpy as np

# отсортируем твиты по “похожести” - чем похожее на source_tweet_index,
# тем ближе к началу списка sorted_similarity
sorted_similarity = np.argsort(-tweet_distance[source_tweet_index,:])
sorted_similarity_num = np.argsort(-tweet_distance)
similarity_values = tweet_distance[source_tweet_index, sorted_similarity]
print(similarity_values)

[1.         0.5118907  0.48507125 0.45834925 0.42443734 0.40422604
 0.38892223 0.38348249 0.37573457 0.37573457 0.36563621 0.36563621
 0.36380344 0.35400522 0.35294118 0.35007002 0.34299717 0.3363364
 0.3363364  0.32410186 0.32338083 0.32338083 0.32338083 0.32338083
 0.31311215 0.31311215 0.306786   0.306786   0.30316953 0.30316953
 0.30316953 0.29704426 0.29411765 0.29411765 0.28005602 0.28005602
 0.28005602 0.28005602 0.28005602 0.27820744 0.27820744 0.27820744
 0.27820744 0.27820744 0.27500955 0.27500955 0.26906912 0.26906912
 0.26906912 0.25928149 0.25928149 0.25928149 0.25928149 0.25928149
 0.25928149 0.25724788 0.25724788 0.25724788 0.25724788 0.25724788
 0.25048972 0.25048972 0.25048972 0.25048972 0.25048972 0.24253563
 0.24253563 0.24253563 0.24253563 0.24253563 0.23529412 0.2300895
 0.2300895  0.2300895  0.2300895  0.22866478 0.21938173 0.21938173
 0.21938173 0.21004201 0.21004201 0.21004201 0.20683508 0.20228869
 0.20180184 0.20180184 0.20180184 0.20180184 0.19446112 0.194461

In [14]:
print(df.iloc[0].tweet_text)
print('-------------')
print(df.iloc[sorted_similarity[1]].tweet_text)
print('-------------')
print(df.iloc[sorted_similarity[2]].tweet_text)
print('-------------')
print(df.iloc[sorted_similarity[3]].tweet_text)

Wow! Google maps for mobile v5 demo at #sxsw. Very nice.
-------------
Apple employees just showed up in force to the #SXSW PopUp Apple Store. #iPad2
-------------
#sxsw apple store run out for the day :( boo apple.
-------------
video from the popup Apple store: {link} #sxsw #sxswi


### Задание высокого уровня.

У вас есть матрица схожести между объектами. Попробуйте решить задачу поиска дубликатов в тексте

1. Визуализируйте гистограмму значений в матрице схожести.
1. Напишите функцию на Python, которая принимает индекс твита, пороговое значение (число от $0.0$ до $1.0$ и матрицу схожести, а затем выводит все твиты, схожесть которых больше, чем пороговое значение.

In [23]:
### YOUR CODE HERE ###
from matplotlib import pyplot as plt

%matplotlib inline

### YOUR CODE HERE ###
def duble_tw(idx, threshold, tweet_distance, df_tweets):
    """
    idx: индекс исходного твита
    threshold: пороговое значение схожести (0.0-1.0)
    tweet_distance: матрица схожести
    df_tweets: датафрейм с твитами (должен быть доступен)
    """
    # Получаем схожесть исходного твита со всеми остальными
    similarities = tweet_distance[idx]

    # Сортируем индексы по убыванию схожести
    sorted_idx = np.argsort(-similarities)

    # Находим твиты, схожесть которых выше порога (исключая сам твит)
    similar_tweets = []
    similar_scores = []

    for i in sorted_idx:
        if i != idx and similarities[i] > threshold:
            similar_tweets.append(df_tweets.iloc[i]['tweet_text'])  # замените на название вашей колонки
            similar_scores.append(similarities[i])

    return similar_tweets, similar_scores





In [26]:
# Использование:
res_tweets, res_scores = duble_tw(14, 0.5, tweet_distance, df)
print(f"Найдено похожих твитов: {len(res_tweets)}")
for i, (tweet, score) in enumerate(zip(res_tweets[:5], res_scores[:5])):
    print(f"{i+1}. Схожесть: {score:.3f}")
    print(df_valid.iloc[14].tweet_text)
    print(f"{tweet[:100]}...\n")

Найдено похожих твитов: 1
1. Схожесть: 0.512
Popup Apple Store crew has been giving out water to the people in line but they are in street clothes. No Apple logos anywhere yet. #SXSW
Apple employees just showed up in force to the #SXSW PopUp Apple Store. #iPad2...

